### **BóSight Week 2: YOLOv8 Detection Training**
#### Trains a YOLOv8s model to detect cows as a single class on MmCows cam_1
#### frames. The workflow was split across two platforms: dataset preparation
#### on Colab CPU and model training on Kaggle T4 GPU.
#
#### Pipeline:
####   1. Copy cam_1 images and labels from Drive to local storage for fast I/O
####   2. Build a temporal-stratified train/val/test split (70/15/15) using
####      5-minute time windows so near-duplicate frames stay together and
####      every split sees both day and night lighting
####   3. Remap cow_id labels (1-16) to single class 0 ("cow") for detection
####   4. Assemble standard YOLOv8 folder layout and data.yaml
####   5. Fine-tune YOLOv8s for 50 epochs
####   6. Evaluate on held-out test set
#
#### **Results:**
####   Val:  P 0.974, R 0.966, mAP50 0.992, mAP50-95 0.765
####  Test: P 0.979, R 0.970, mAP50 0.992, mAP50-95 0.778

#### Note: Part 1 paths assume Colab with Google Drive mounted at /content/drive.
#### Part 2 paths assume Kaggle, including a dataset path under a specific
#### username; update DATA_PATH to your own Kaggle dataset location.

#### Part 1: Dataset Preparation
Platform : Colab (CPU)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports and copy data to local storage.
# Copying from Drive to local /content/ bypasses Drive's per-file open
# latency, which significantly slows training and data loading.
import os
import shutil
import random
import json
from pathlib import Path
from collections import defaultdict

In [ ]:
# Paths to MmCows cam_1 data on Drive
DRIVE_IMAGES_DIR = Path("/content/drive/MyDrive/MmCows/extracted/visual_data/images/0725/cam_1")
DRIVE_LABELS_DIR = Path("/content/drive/MyDrive/MmCows/extracted/visual_data/labels/combined/0725/cam_1")
LOCAL_ROOT = Path("/content/bosight_local")

In [ ]:
# Fixed seed for reproducible splits
SEED = 42
random.seed(SEED)

In [ ]:
# Copy images and labels to local storage (skips if already done)
local_images = LOCAL_ROOT / "images"
local_labels = LOCAL_ROOT / "labels"

if not local_images.exists():
    print("Copying images...")
    shutil.copytree(DRIVE_IMAGES_DIR, local_images)
if not local_labels.exists():
    print("Copying labels...")
    shutil.copytree(DRIVE_LABELS_DIR, local_labels)

# Verify: should be 5040 each
print(f"Images: {len(list(local_images.glob('*.jpg')))}")
print(f"Labels: {len(list(local_labels.glob('*.txt')))}")

In [ ]:
# Temporal-stratified split + YOLOv8 dataset assembly.
# Split strategy: group frames into 5-minute time windows, then assign
# whole windows to train/val/test. This prevents near-duplicate frames
# (captured ~17s apart) from leaking across splits. The assignment is
# stratified by hour-of-day so every split sees both daytime and nighttime.

WINDOW_SECONDS = 300  # 5-minute windows
DATASET_ROOT = Path("/content/bosight_dataset")

def parse_stem(stem):
    """Extract Unix timestamp and hour from filename stem.
    Filename format: '{unix_ts}_{HH-MM-SS}' e.g. '1690332506_19-48-26'
    """
    ts_str, hms = stem.split("_", 1)
    return int(ts_str), int(hms.split("-")[0])

# Get all image stems sorted chronologically
stems = sorted(p.stem for p in local_images.glob("*.jpg"))

# Group frames into 5-minute windows
window_to_stems = defaultdict(list)
window_to_hour = {}
for stem in stems:
    ts, hour = parse_stem(stem)
    wid = ts // WINDOW_SECONDS
    window_to_stems[wid].append(stem)
    window_to_hour[wid] = hour

# Group windows by hour for stratification
hour_to_windows = defaultdict(list)
for wid, hour in window_to_hour.items():
    hour_to_windows[hour].append(wid)

# Assign windows to splits within each hour (70/15/15)
split_stems = {"train": [], "val": [], "test": []}
for hour, windows in sorted(hour_to_windows.items()):
    windows = windows[:]
    random.shuffle(windows)
    n = len(windows)
    n_train = round(n * 0.70)
    n_val = round(n * 0.15)
    assign = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)
    for wid, split in zip(windows, assign):
        split_stems[split].extend(window_to_stems[wid])

# Assemble YOLOv8 folder layout: images/{split}/ and labels/{split}/
# Remap cow_id (1-16) to class 0 ("cow"), detection doesn't need identity
for split, stems_list in split_stems.items():
    img_out = DATASET_ROOT / "images" / split
    lbl_out = DATASET_ROOT / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for stem in stems_list:
        # Hard-link image (instant, no disk cost) with copy fallback
        src_img = local_images / f"{stem}.jpg"
        dst_img = img_out / f"{stem}.jpg"
        if not dst_img.exists():
            try:
                os.link(src_img, dst_img)
            except OSError:
                shutil.copy2(src_img, dst_img)

        # Read original label and remap: cow_id x y w h -> 0 x y w h
        src_lbl = local_labels / f"{stem}.txt"
        remapped = []
        if src_lbl.exists():
            for line in open(src_lbl):
                parts = line.strip().split()
                if parts:
                    remapped.append(f"0 {parts[1]} {parts[2]} {parts[3]} {parts[4]}")
        with open(lbl_out / f"{stem}.txt", "w") as f:
            f.write("\n".join(remapped))

# Write YOLOv8 config file
(DATASET_ROOT / "bosight.yaml").write_text(
    f"path: {DATASET_ROOT}\ntrain: images/train\nval: images/val\n"
    f"test: images/test\nnc: 1\nnames: ['cow']\n"
)

# Save split manifest for reproducibility
with open(DATASET_ROOT / "split_manifest.json", "w") as f:
    json.dump(split_stems, f, indent=2)

# Verify bbox counts per split
# Expected: train ~35269, val ~9267, test ~8972
for split in ["train", "val", "test"]:
    lbl_dir = DATASET_ROOT / "labels" / split
    total = sum(len([l for l in f.read_text().strip().split("\n") if l.strip()])
                for f in lbl_dir.glob("*.txt"))
    print(f"{split}: {len(list(lbl_dir.glob('*.txt')))} files, {total} bbox lines")

In [ ]:
# Zip dataset for Kaggle upload.
# The zipped dataset is uploaded to Kaggle as a dataset, then attached
# to the training notebook. This avoids re-running the split on Kaggle.
# !zip -r -0 "/content/drive/MyDrive/MmCows/bosight_dataset.zip" /content/bosight_dataset

#### Part 2: Model Training
Platform : Kaggle (T4 GPU)

In [ ]:
# Requires the zipped dataset from Part 1 to be manually uploaded to Kaggle as a dataset first.
from pathlib import Path
from ultralytics import YOLO

DATA_PATH = "/kaggle/input/datasets/anandhvenkataraman/mmcows/content/bosight_dataset"
yaml_path = "/kaggle/working/bosight.yaml"

Path(yaml_path).write_text(
    f"path: {DATA_PATH}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n"
    f"nc: 1\n"
    f"names: ['cow']\n"
)

# YOLOv8s: small variant, good balance of speed and accuracy
# device=0 forces GPU usage (without this, Ultralytics may default to CPU)
model = YOLO("yolov8s.pt")

results = model.train(
    data=yaml_path,
    epochs=50,         # max epochs; patience may stop earlier
    imgsz=640,         # resize from 4480x2800 to 640px; sufficient for cow detection
    batch=8,           # fits within T4's 15GB VRAM
    workers=2,         # data loader workers; higher values may hang on Kaggle
    cache=False,       # don't cache images in RAM (prevents OOM)
    device=0,          # explicitly use GPU
    name="bosight_det_v3",
    project="/kaggle/working/bosight_runs",
    seed=42,           # reproducibility
    patience=10,       # early stop if val mAP doesn't improve for 10 epochs
    save=True,
    save_period=5,     # checkpoint every 5 epochs
    plots=True,        # save training curves and confusion matrix
)

# %%
# Test set evaluation.
# Evaluate best.pt on the held-out test split (840 images, never seen in training)
model = YOLO("/kaggle/working/bosight_runs/bosight_det_v3/weights/best.pt")

results = model.val(
    data=yaml_path,
    split="test",
    device=0,
    plots=True,
)

In [ ]:
# Test set evaluation.
# Evaluate best.pt on the held-out test split (840 images, never seen in training)
model = YOLO("/kaggle/working/bosight_runs/bosight_det_v3/weights/best.pt")

results = model.val(
    data=yaml_path,
    split="test",
    device=0,
    plots=True,
)

In [ ]:
# Download weights.
# Copy best.pt to /kaggle/working/ so it appears in the Output tab
# after Save Version. Download and upload to Drive.
from IPython.display import FileLink
display(FileLink("/kaggle/working/bosight_runs/bosight_det_v3/weights/best.pt"))